In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

df = pd.read_sas("../data_raw/LLCP2015.XPT")
print(f"Raw data shape: {df.shape}")
print(f"Columns: {df.shape[1]}")
print(f"Rows: {df.shape[0]:,}")

Raw data shape: (441456, 330)
Columns: 330
Rows: 441,456


In [ ]:
## Step 2 — Create Heart Disease Target Variable

Combine two BRFSS columns:
- `CVDINFR4` → Heart attack history (1 = Yes)
- `CVDCRHD4` → Coronary heart disease (1 = Yes)

**Target:** `heart_disease` = 1 if either condition is present, 0 otherwise.

In [2]:
# Preview raw heart disease columns
print("Raw value counts:")
print("CVDINFR4 (Heart attack):", dict(df["CVDINFR4"].value_counts().head()))
print("CVDCRHD4 (Coronary HD):", dict(df["CVDCRHD4"].value_counts().head()))

# Create binary target: 1 = heart disease present, 0 = no heart disease
df["heart_disease"] = (
    (df["CVDINFR4"] == 1) |
    (df["CVDCRHD4"] == 1)
).astype(int)

print(f"\nTarget distribution:")
print(df["heart_disease"].value_counts())
print(f"\nHeart disease prevalence: {df['heart_disease'].mean():.3f} ({df['heart_disease'].mean()*100:.1f}%)")

Raw value counts:
CVDINFR4 (Heart attack): {2.0: np.int64(413755), 1.0: np.int64(25472), 7.0: np.int64(2038), 9.0: np.int64(191)}
CVDCRHD4 (Coronary HD): {2.0: np.int64(412349), 1.0: np.int64(25290), 7.0: np.int64(3591), 9.0: np.int64(225)}

Target distribution:
heart_disease
0    402823
1     38633
Name: count, dtype: int64

Heart disease prevalence: 0.088 (8.8%)


In [ ]:
## Step 3 — Select Features

14 medically relevant predictors for heart disease risk:

| Feature | Description |
|---|---|
| `_AGEG5YR` | Age group (1–13) |
| `SEX` | Gender (1=Male, 2=Female) |
| `_BMI5` | Body mass index (×100) |
| `_RFHYPE5` | High blood pressure |
| `_RFCHOL` | High cholesterol |
| `SMOKE100` | Smoking history (100+ cigarettes) |
| `_TOTINDA` | Physical activity |
| `_FRTLT1` | Fruit consumption |
| `_VEGLT1` | Vegetable consumption |
| `_RFDRHV5` | Heavy alcohol drinking |
| `GENHLTH` | General health rating (1–5) |
| `MENTHLTH` | Mental health days (0–30) |
| `PHYSHLTH` | Physical health days (0–30) |
| `DIABETE3` | Diabetes diagnosis |

(441456, 330)
   CVDINFR4  CVDCRHD4  CVDSTRK3
0       2.0       2.0       2.0
1       2.0       2.0       2.0
2       7.0       2.0       1.0
3       2.0       2.0       2.0
4       2.0       2.0       2.0


In [3]:
features = [
    "_AGEG5YR",   # Age group
    "SEX",        # Gender
    "_BMI5",      # Body mass index
    "_RFHYPE5",   # High blood pressure
    "_RFCHOL",    # High cholesterol
    "SMOKE100",   # Smoking history
    "_TOTINDA",   # Physical activity
    "_FRTLT1",    # Fruit consumption
    "_VEGLT1",    # Vegetable consumption
    "_RFDRHV5",   # Heavy alcohol drinking
    "GENHLTH",    # General health rating
    "MENTHLTH",   # Mental health days
    "PHYSHLTH",   # Physical health days
    "DIABETE3",   # Diabetes diagnosis
]

target = "heart_disease"
df_model = df[features + [target]].copy()

print(f"Selected features: {len(features)}")
print(f"Model dataframe shape: {df_model.shape}")
df_model.head()

Selected features: 14
Model dataframe shape: (441456, 15)


,_AGEG5YR,SEX,_BMI5,_RFHYPE5,_RFCHOL,SMOKE100,_TOTINDA,_FRTLT1,_VEGLT1,_RFDRHV5,GENHLTH,MENTHLTH,PHYSHLTH,DIABETE3,heart_disease
0,9.0,2.0,4018.0,2.0,2.0,1.0,2.0,2.0,1.0,1.0,5.0,18.0,15.0,3.0,0
1,7.0,2.0,2509.0,1.0,1.0,1.0,1.0,2.0,2.0,1.0,3.0,88.0,88.0,3.0,0
2,11.0,2.0,2204.0,1.0,2.0,NaN,9.0,9.0,9.0,9.0,4.0,88.0,15.0,3.0,0
3,9.0,2.0,2819.0,2.0,2.0,2.0,2.0,1.0,2.0,1.0,5.0,30.0,30.0,3.0,0
4,9.0,2.0,2437.0,1.0,1.0,2.0,2.0,9.0,1.0,1.0,5.0,88.0,20.0,3.0,0


In [ ]:
## Step 4 — Missing Value Handling

BRFSS uses special codes for missing data:
- `7` = "Don't know / Not sure"
- `9` = "Refused"

Replace these with `NaN` and drop rows with any remaining missing values.

In [4]:
print(f"Before cleaning: {df_model.shape}")
print(f"Missing values per column:\n{df_model.isnull().sum()}\n")

# Replace BRFSS missing-value codes
df_model = df_model.replace({7: None, 9: None})

# Drop rows with any NaN
df_model = df_model.dropna()

print(f"After cleaning: {df_model.shape}")
print(f"Rows dropped: {df.shape[0] - df_model.shape[0]:,}")

# Validation: dataset should have >200k rows
assert df_model.shape[0] > 200_000, f"Dataset too small: {df_model.shape[0]} rows"
print(f"\n✓ Validation passed: {df_model.shape[0]:,} rows (>200k required)")

Before cleaning: (441456, 15)
Missing values per column:
_AGEG5YR             0
SEX                  0
_BMI5            36398
_RFHYPE5             0
_RFCHOL          59154
SMOKE100         14255
_TOTINDA             0
_FRTLT1              0
_VEGLT1              0
_RFDRHV5             0
GENHLTH              2
MENTHLTH             0
PHYSHLTH             1
DIABETE3             7
heart_disease        0
dtype: int64

After cleaning: (228102, 15)
Rows dropped: 213,354

✓ Validation passed: 228,102 rows (>200k required)


In [ ]:
## Step 5 — Feature Engineering

- **BMI:** BRFSS stores BMI × 100 (e.g., 4018 → 40.18). Divide by 100.
- **Binary features:** BRFSS uses 1=Yes, 2=No. Recode to 1/0.
- **Gender:** 1=Male, 2=Female → 1=Male, 0=Female.
- **DIABETE3:** 1=Yes, 3=No, 2=Yes (pregnancy), 4=Borderline → recode to binary.
- **MENTHLTH / PHYSHLTH:** 88 = "None" → recode to 0.

(441456, 15)

In [5]:
# ── BMI: scale from ×100 ─────────────────────────────────────────
df_model["_BMI5"] = df_model["_BMI5"] / 100

# ── Binary features: BRFSS 1=Yes, 2=No → 1/0 ──────────────────
binary_cols = ["_RFHYPE5", "_RFCHOL", "SMOKE100", "_TOTINDA", "_FRTLT1", "_VEGLT1", "_RFDRHV5"]
for col in binary_cols:
    df_model[col] = df_model[col].map({1: 1, 2: 0})

# ── Gender: 1=Male, 2=Female → 1/0 ─────────────────────────────
df_model["SEX"] = df_model["SEX"].map({1: 1, 2: 0})

# ── Diabetes: 1=Yes, 3=No, 2=Pregnancy, 4=Borderline → binary ──
df_model["DIABETE3"] = df_model["DIABETE3"].map({1: 1, 2: 1, 3: 0, 4: 1})

# ── Mental/Physical health: 88="None" → 0, 77/99 → NaN ─────────
for col in ["MENTHLTH", "PHYSHLTH"]:
    df_model[col] = df_model[col].replace({88: 0, 77: pd.NA, 99: pd.NA})

# ── Age group: 14="Don't know" → drop ──────────────────────────
df_model["_AGEG5YR"] = df_model["_AGEG5YR"].replace(14, pd.NA)

# Drop any new NaNs from recoding
df_model = df_model.dropna()

print(f"After feature engineering: {df_model.shape}")
print(f"\nFeature value ranges:")
for col in features:
    print(f"  {col:.<20} min={df_model[col].min():.1f}  max={df_model[col].max():.1f}")

After feature engineering: (220877, 15)

Feature value ranges:
  _AGEG5YR............ min=1.0  max=13.0
  SEX................. min=0.0  max=1.0
  _BMI5............... min=12.1  max=97.7
  _RFHYPE5............ min=0.0  max=1.0
  _RFCHOL............. min=0.0  max=1.0
  SMOKE100............ min=0.0  max=1.0
  _TOTINDA............ min=0.0  max=1.0
  _FRTLT1............. min=0.0  max=1.0
  _VEGLT1............. min=0.0  max=1.0
  _RFDRHV5............ min=0.0  max=1.0
  GENHLTH............. min=1.0  max=5.0
  MENTHLTH............ min=0.0  max=30.0
  PHYSHLTH............ min=0.0  max=30.0
  DIABETE3............ min=0.0  max=1.0


In [ ]:
## Step 6 — Data Validation

In [6]:
X = df_model.drop(columns=["heart_disease"])
y = df_model["heart_disease"]

# Ensure all features are numeric
X = X.apply(pd.to_numeric, errors="coerce")

print("=" * 50)
print("DATA VALIDATION CHECKS")
print("=" * 50)

# Check 1: Dataset size
print(f"\n1. Dataset size: {X.shape[0]:,} rows, {X.shape[1]} features")
assert X.shape[0] > 200_000, "Dataset too small"
print("   ✓ >200k rows")

# Check 2: Feature count
assert X.shape[1] == 14, f"Expected 14 features, got {X.shape[1]}"
print(f"   ✓ 14 features")

# Check 3: No missing values
assert X.isnull().sum().sum() == 0, "Missing values found"
print("   ✓ No missing values")

# Check 4: All numeric types
non_numeric = X.select_dtypes(exclude=["int", "float", "bool"]).columns.tolist()
assert len(non_numeric) == 0, f"Non-numeric columns: {non_numeric}"
print("   ✓ All features numeric")

# Check 5: Target distribution
print(f"\n2. Target distribution:")
print(f"   Healthy (0): {(y == 0).sum():,}")
print(f"   Disease (1): {(y == 1).sum():,}")
print(f"   Imbalance ratio: {(y == 0).sum() / (y == 1).sum():.1f} : 1")
print(f"   Prevalence: {y.mean():.3f} ({y.mean()*100:.1f}%)")

# Check 6: Data types
print(f"\n3. Data types:")
print(X.dtypes.value_counts().to_string())

DATA VALIDATION CHECKS

1. Dataset size: 220,877 rows, 14 features
   ✓ >200k rows
   ✓ 14 features
   ✓ No missing values
   ✓ All features numeric

2. Target distribution:
   Healthy (0): 198,813
   Disease (1): 22,064
   Imbalance ratio: 9.0 : 1
   Prevalence: 0.100 (10.0%)

3. Data types:
int64      11
float64     3


In [ ]:
## Step 7 — Train / Test Split

Stratified 80/20 split to preserve class distribution.

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

# Ensure float64 for consistency
X_train = X_train.astype(np.float64)
X_test = X_test.astype(np.float64)
y_train = y_train.astype(np.float64)
y_test = y_test.astype(np.float64)

# Compute scale_pos_weight for class imbalance
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f"Train: {X_train.shape}   Test: {X_test.shape}")
print(f"Positive (disease=1): {pos:,}")
print(f"Negative (healthy=0): {neg:,}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

Train: (176701, 14)   Test: (44176, 14)
Positive (disease=1): 17,651
Negative (healthy=0): 159,050
scale_pos_weight: 9.01


In [ ]:
## Step 8 — Train XGBoost with Class Imbalance Handling

Using `scale_pos_weight` to compensate for the ~9:1 class imbalance (healthy vs disease). Early stopping prevents overfitting.

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=30,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,
)

print(f"\nBest iteration: {model.best_iteration}")
print(f"Best AUC: {model.best_score:.4f}")

In [ ]:
## Step 9 — Evaluate Model Performance

Assess the trained model on the held-out test set using:
- **ROC-AUC** — area under the receiver-operating-characteristic curve
- **Classification report** — precision, recall, F1 for each class
- **Confusion matrix** — counts of TP, FP, TN, FN

Because the dataset is imbalanced, we care most about **ROC-AUC** and **recall for the positive class** (catching actual heart-disease cases).

_AGEG5YR    object
SEX          int64
_BMI5       object
_RFHYPE5     int64
_RFCHOL      int64
SMOKE100     int64
_TOTINDA     int64
_FRTLT1      int64
_VEGLT1      int64
_RFDRHV5     int64
GENHLTH     object
MENTHLTH    object
PHYSHLTH    object
DIABETE3    object
dtype: object

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC: {auc:.4f}\n")
print(classification_report(y_test, y_pred, target_names=["No Heart Disease", "Heart Disease"]))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ax.matshow(cm, cmap="Blues", alpha=0.7)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=14)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["No HD", "HD"])
ax.set_yticklabels(["No HD", "HD"])
plt.tight_layout()
plt.show()

In [ ]:
## Step 10 — Feature Importance Validation

Inspect the top features by XGBoost's built-in "gain" importance to verify they are **medically sensible**. Expected top contributors for heart disease include age, high blood pressure, high cholesterol, BMI, and general health status.

In [ ]:
importances = model.get_booster().get_score(importance_type="gain")
imp_df = (
    pd.DataFrame.from_dict(importances, orient="index", columns=["gain"])
    .sort_values("gain", ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 6))
imp_df.plot.barh(ax=ax, legend=False, color="steelblue")
ax.set_title("Feature Importance (Gain)")
ax.set_xlabel("Mean Gain")
plt.tight_layout()
plt.show()

print("\nTop 5 features:")
for feat, gain in imp_df.sort_values("gain", ascending=False).head().iterrows():
    print(f"  {feat}: {gain['gain']:.1f}")

_AGEG5YR    float64
SEX           int64
_BMI5       float64
_RFHYPE5      int64
_RFCHOL       int64
SMOKE100      int64
_TOTINDA      int64
_FRTLT1       int64
_VEGLT1       int64
_RFDRHV5      int64
GENHLTH     float64
MENTHLTH    float64
PHYSHLTH    float64
DIABETE3    float64
dtype: object

In [ ]:
## Step 11 — Isotonic Probability Calibration

XGBoost with `scale_pos_weight` inflates predicted probabilities (mean ~0.35 vs actual prevalence ~0.10). We use **isotonic regression** to map raw probabilities to well-calibrated ones.

This matches the diabetes model's approach — a separate `IsotonicRegression` calibrator saved alongside the model.

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [ ]:
from sklearn.isotonic import IsotonicRegression

calibrator = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip")
calibrator.fit(y_prob, y_test)

cal_prob = calibrator.predict(y_prob)
cal_auc = roc_auc_score(y_test, cal_prob)

print(f"Raw probabilities   — mean: {y_prob.mean():.4f}, std: {y_prob.std():.4f}")
print(f"Calibrated probs    — mean: {cal_prob.mean():.4f}, std: {cal_prob.std():.4f}")
print(f"Actual positive rate: {y_test.mean():.4f}")
print(f"Calibrated AUC: {cal_auc:.4f}")

              precision    recall  f1-score   support

           0       0.90      0.99      0.95     40998
           1       0.53      0.07      0.12      4623

    accuracy                           0.90     45621
   macro avg       0.72      0.53      0.53     45621
weighted avg       0.87      0.90      0.86     45621

ROC-AUC: 0.8406026849444453


In [ ]:
## Step 12 — Post-Calibration Evaluation

Re-evaluate using the calibrated model to confirm that calibration did not degrade discriminative performance (ROC-AUC should remain similar).

_RFHYPE5    0.220799
GENHLTH     0.215658
_AGEG5YR    0.172606
_RFCHOL     0.112345
SEX         0.111720
SMOKE100    0.040578
DIABETE3    0.039397
PHYSHLTH    0.018160
_RFDRHV5    0.016368
_BMI5       0.011662
_TOTINDA    0.010808
MENTHLTH    0.010775
_FRTLT1     0.009680
_VEGLT1     0.009444
dtype: float32

In [ ]:
cal_pred = (cal_prob >= 0.5).astype(int)

print(f"Pre-calibration  ROC-AUC: {auc:.4f}")
print(f"Post-calibration ROC-AUC: {cal_auc:.4f}\n")
print(classification_report(y_test, cal_pred, target_names=["No Heart Disease", "Heart Disease"]))

,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2","XGBClassifier...ree=None, ...)"
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'isotonic'
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors.Base estimator clones are fitted in parallel across cross-validationiterations.See :term:`Glossary ` for more details... versionadded:: 0.24",None
,"ensemble ensemble: bool, or ""auto"", default=""auto""Determines how the calibrator is fitted.""auto"" will use `False` if the `estimator` is a:class:`~sklearn.frozen.FrozenEstimator`, and `True` otherwise.If `True`, the `estimator` is fitted using training data, andcalibrated using testing data, for each `cv` fold. The final estimatoris an ensemble of `n_cv` fitted classifier and calibrator pairs, where`n_cv` is the number of cross-validation folds. The output is theaverage predicted probabilities of all pairs.If `False`, `cv` is used to compute unbiased predictions, via:func:`~sklearn.model_selection.cross_val_predict`, which are thenused for calibration. At prediction time, the classifier used is the`estimator` trained on all the data.Note that this method is also internally implemented in:mod:`sklearn.svm` estimators with the `probabilities=True` parameter... versionadded:: 0.24.. versionchanged:: 1.6 `""auto""` option is added and is the default.",'auto'
,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[fl

In [ ]:
## Step 13 — Save Model Artifacts

Export the XGBoost model, isotonic calibrator, and feature list for FastAPI inference.

| Artifact | Path | Contents |
|----------|------|----------|
| XGBoost model | `models/heart_disease_xgboost.pkl` | Trained `XGBClassifier` |
| Calibrator | `models/heart_disease_calibrator.pkl` | `IsotonicRegression` for probability calibration |
| Feature list | `models/heart_disease_features.pkl` | Ordered list of 14 column names |

heart_disease
0    204988
1     23114
Name: count, dtype: int64

In [ ]:
import joblib, os

models_dir = os.path.join("..", "models")
os.makedirs(models_dir, exist_ok=True)

model_path = os.path.join(models_dir, "heart_disease_xgboost.pkl")
calibrator_path = os.path.join(models_dir, "heart_disease_calibrator.pkl")
features_path = os.path.join(models_dir, "heart_disease_features.pkl")

joblib.dump(model, model_path)
joblib.dump(calibrator, calibrator_path)
joblib.dump(X.columns.tolist(), features_path)

print(f"Saved model      → {model_path}  ({os.path.getsize(model_path) / 1024:.0f} KB)")
print(f"Saved calibrator → {calibrator_path}  ({os.path.getsize(calibrator_path) / 1024:.0f} KB)")
print(f"Saved features   → {features_path}")
print(f"Features ({len(X.columns)}): {X.columns.tolist()}")

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [ ]:
## Step 14 — Quick Sanity Check

Reload the saved model and run a test prediction to verify the artifact works end-to-end.

              precision    recall  f1-score   support

           0       0.97      0.69      0.81     40998
           1       0.23      0.83      0.37      4623

    accuracy                           0.71     45621
   macro avg       0.60      0.76      0.59     45621
weighted avg       0.90      0.71      0.76     45621

ROC-AUC: 0.8399274859506027


In [ ]:
# Reload and verify
loaded_model = joblib.load(model_path)
loaded_calibrator = joblib.load(calibrator_path)
loaded_features = joblib.load(features_path)

# Test with a sample from the test set
sample = X_test.iloc[[0]]
raw_prob = loaded_model.predict_proba(sample)[:, 1][0]
cal_prob_val = loaded_calibrator.predict([raw_prob])[0]
risk = "High" if cal_prob_val >= 0.6 else ("Moderate" if cal_prob_val >= 0.3 else "Low")

print("=== Sanity Check ===")
print(f"Sample features: {sample.values[0]}")
print(f"Raw probability:        {raw_prob:.4f}")
print(f"Calibrated probability: {cal_prob_val:.4f}")
print(f"Risk category: {risk}")
print(f"\n✅ Heart disease model pipeline complete!")
print(f"   Model:      {model_path}")
print(f"   Calibrator: {calibrator_path}")
print(f"   Features:   {loaded_features}")